# BHT Agentic Pipeline Notebook (Anaconda + Azure OpenAI)

This notebook shows how to read an SPSS `.sav` file and run `bht_agentic_pipeline.py` using Azure OpenAI settings from an environment file (`.env` or `.evn`).

In [ ]:
# If needed once in this kernel
# !pip install pyreadstat pandas requests python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_file = Path('.env')
if not env_file.exists() and Path('.evn').exists():
    env_file = Path('.evn')  # supports your requested filename typo as well

load_dotenv(env_file if env_file.exists() else None)
print('Loaded env file:', env_file if env_file.exists() else 'None found')

In [ ]:
# Required Azure env vars
required = [
    'AZURE_OPENAI_API_KEY',
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_OPENAI_DEPLOYMENT',
    'AZURE_OPENAI_API_VERSION',
]
for k in required:
    print(k, 'OK' if os.getenv(k) else 'MISSING')

In [ ]:
import pyreadstat
sav_path = './data/your_file.sav'  # <-- update this
df, meta = pyreadstat.read_sav(sav_path, apply_value_formats=False)
print('Rows:', len(df), 'Cols:', len(df.columns))
df.head(3)

In [ ]:
import subprocess
cmd = [
    'python', 'bht_agentic_pipeline.py', sav_path,
    '--outdir', 'outputs_notebook',
    '--provider', 'azure'
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

In [ ]:
import json
from pathlib import Path
out = Path('outputs_notebook')
for name in ['master_metadata.json', 'column_mapping.json', 'questionnaire_logic.json']:
    p = out / name
    print(name, 'exists:', p.exists())

sample = json.loads((out / 'master_metadata.json').read_text())
list(sample.items())[:5]